In [ ]:
#!pip install bitsandbytes

In [1]:
from transformers import (
    AutoTokenizer,
    AutoModel,
    AutoModelForCausalLM,
    pipeline
)
import torch

from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

### Load Generation Model and Tokenizer

In [2]:
model_name = "unsloth/medgemma-4b-it-bnb-4bit"

print(f"🔄 Loading generation model: {model_name}")

generation_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
    device_map="auto"
)
generation_tokenizer = AutoTokenizer.from_pretrained(model_name)
generation_tokenizer.pad_token = generation_tokenizer.pad_token or generation_tokenizer.eos_token

🔄 Loading generation model: unsloth/medgemma-4b-it-bnb-4bit


The installed version of bitsandbytes was compiled without GPU support. 8-bit optimizers and GPU quantization are unavailable.
/anaconda/envs/mlcourse_env/lib/python3.10/site-packages/torch/cuda/__init__.py:789: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")


model.safetensors:   0%|          | 0.00/3.23G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/210 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/4.69M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/670 [00:00<?, ?B/s]

In [3]:
def generate(messages):

    text = generation_tokenizer.apply_chat_template(messages, add_generation_prompt=True, tokenize=False)
    inputs = generation_tokenizer(text, return_tensors="pt", padding=True, truncation=True, max_length=1000).to(generation_model.device)

    with torch.inference_mode():
        generation = generation_model.generate(
            **inputs, max_new_tokens=300, do_sample=False,
            pad_token_id=generation_tokenizer.eos_token_id
        )

    output = generation_tokenizer.decode(generation[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)

    return output

### Load Notes

In [ ]:
note = """The patient is a 72-year-old female with a past medical history of atrial fibrillation, chronic kidney disease stage 3, hypertension, and osteoarthritis. 
She presented to the ED with complaints of generalized weakness, lightheadedness, and two episodes of syncope over the past 24 hours.

On exam, she was hypotensive (BP 88/56) and bradycardic with an irregularly irregular pulse. Labs showed elevated BUN and creatinine, consistent with worsening CKD, and mild hyperkalemia. 
ECG revealed atrial fibrillation with slow ventricular response. Chest X-ray showed no acute pulmonary findings.
The patient was admitted to telemetry for further monitoring. Her beta-blocker (metoprolol) was held, and nephrology was consulted for worsening renal function. C
ardiology was consulted for bradyarrhythmia management. A temporary transvenous pacemaker was placed.

She was monitored closely, and after stabilization, underwent permanent pacemaker implantation on hospital day 3. Post-op course was uncomplicated. 
Renal function improved with IV fluids and adjustment of antihypertensive medications.

Discharge medications included: apixaban, lisinopril (dose reduced), and acetaminophen for pain. She was advised to follow up with cardiology and nephrology within one week.
Discharged home with stable vital signs and improved energy levels."""

In [4]:
note_pt = """1.a consulta
NHC 118441566
H, 67 anos, de Sesimbra, mestre pesca reformado (desde 58 anos)
Sempre trabalhou de noite e dormiu de dia. Dormia cerca de 3 h.
Desde que se reformou tem dificuldade em dormir. Dorme 3-4 h
Tem uma empresa. Por vezes ansioso
Faz alprazolan 0,25 mg ao jantar. Deita-se 21h-22h. Leva cerca de 2 horas para adormecer. Dorme 3-4 h. ACorda e não sabe porquê. Fica na cama e depois levanta-se.
Antes
Já fez mirtazapina 15 mg, durante 1 Mês e acha que não teve efeito terapeutico
DRC por provável pielonefrite crónica litiásica (hiperuricémia?), inicio de hemodiálise em julho 2002, Transplante renal de dador vivo a 17/09/2002. Reinicio de hemodiálise por disfunção crónica do enxerto a 17/08/2016
HTA secundária
Hábitos tabágicos suspensos 2002

Adenocrcinoma da prostata, submetido a prostatectomia total em janeiro de 2018

Tumor Septissémia por bactéria provocada por um peixe... em set 2018. Ferida no MS dto...

Medicação atual: amlodipina e concor . Alprazolam

Refere sensação de calor nos membros inferiores que melhora com movimento, piora no final do dia, por vezes dificultam o adormecer

Roncopatia. Sem apneias identificadas.

Xavier Mourato OM 52847"""

## Summarization

In [6]:
messages = [
    {
        "role": "system",
        "content": "You are a helpful medical assistant."
    },
    {
        "role": "user",
        "content": f"""Summarize the following clinical note:\n\nCLINICAL NOTE:\n{note2}"""
    }
]

output = generate(messages)
print(output)

The following generation flags are not valid and may be ignored: ['top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


KeyboardInterrupt: 

## Entity Extraction

### With a LLM

In [52]:
messages = [
    {
        "role": "system",
        "content": "You are a helpful medical assistant."
    },
    {
        "role": "user",
        "content": f"""Extract the diagnosis found in the following clinical note:\n\nCLINICAL NOTE:\n{note_pt}"""
    }
]

output = generate(messages)
print(output)

The following generation flags are not valid and may be ignored: ['top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


The diagnosis is:

*   **Adenocarcinoma da prostata** (Prostate cancer)
*   **Pielonefrite crónica litiásica (hiperuricémia?)** (Chronic lithiatic pyelonephritis (hyperuricemia?))
*   **HTA secundária** (Secondary hypertension)
*   **Roncopatia** (Snoring)


### With MediAlbertina

In [44]:
entity_extractor = pipeline('ner', model='portugueseNLP/medialbertina_pt-pt_900m_NER', aggregation_strategy="simple")
entities = entity_extractor(note_pt)

Device set to use cuda:0


In [49]:
diagnosis = [annotation['word'] for annotation in entities if annotation['entity_group'] == 'Diagnostico']

In [50]:
diagnosis

['DRC',
 'pielonefrite crónica litiásica (hiperuricémia?',
 'disfunção crónica do enxerto',
 'HTA secundária',
 'Adenocrcinoma da prostata',
 'Tumor Septissémia',
 'bactéria provocada por um peixe...',
 'Ferida no MS dto',
 'Roncopatia',
 'apneias']